# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

from uavloc.features.segmentation import class_probabilities, load_segmenter
from uavloc.segmentation.eval import load_val_sets
from uavloc.segmentation.losses import TRACK, class_skeleton
from uavloc.track import REPO_ROOT, resolve_device

# Params

In [ ]:
VAL_SET = "bushcha"
DATE = "2024-06-06"
ARCHITECTURE = "unet-resnet34"
LOSSES = ("current", "structure")
SEED = 42
STEPS = 2500
SKELETON_DILATION_PX = 1
WINDOW_PX = 512
WINDOW_STEP_PX = 256
ZOOM_PX = 96
DEVICE = "auto"
DPI = 200
MASK_ALPHA = 0.45
EXP_DIR = REPO_ROOT / "experiments" / "021-building-loss-from-scratch"
OUT_DIR = EXP_DIR / "figures" / "skeleton-recall"

In [ ]:
device = resolve_device(DEVICE)
OUT_DIR.mkdir(parents=True, exist_ok=True)
device

# Data

In [ ]:
image = next(a for a in load_val_sets()[VAL_SET] if a.date == DATE)
image.rgb.shape

In [ ]:
labels = torch.from_numpy(image.label.astype(np.int64))[None]
tube = class_skeleton(labels, TRACK, SKELETON_DILATION_PX)[0].numpy()
centreline = class_skeleton(labels, TRACK, 0)[0].numpy()
track = image.label == TRACK
int(tube.sum()), int(centreline.sum())

# Model

In [ ]:
def checkpoint(loss: str):
    return EXP_DIR / "artifacts" / f"model-{ARCHITECTURE}-{loss}-seed{SEED}-steps{STEPS}.pt"

In [ ]:
def track_probability(loss: str) -> np.ndarray:
    """`(H, W)` softmax probability of the track class on the whole image."""
    model = load_segmenter(checkpoint(loss), device, ARCHITECTURE)
    return class_probabilities(model, image.rgb, device)[TRACK]

In [ ]:
p_track = {loss: track_probability(loss) for loss in LOSSES}

# Run

In [ ]:
def skeleton_recall_loss(p: np.ndarray, skeleton: np.ndarray) -> float:
    return float(1 - (p * skeleton).sum() / skeleton.sum())

In [ ]:
def densest_window(skeleton: np.ndarray) -> tuple[int, int]:
    """Top-left of the `WINDOW_PX` window with the most skeleton pixels, on a `WINDOW_STEP_PX` grid."""
    h, w = skeleton.shape
    corners = [(r, c) for r in range(0, h - WINDOW_PX + 1, WINDOW_STEP_PX)
               for c in range(0, w - WINDOW_PX + 1, WINDOW_STEP_PX)]
    return max(corners, key=lambda rc: skeleton[rc[0]:rc[0] + WINDOW_PX, rc[1]:rc[1] + WINDOW_PX].sum())

In [ ]:
row, col = densest_window(tube)
window = np.s_[row:row + WINDOW_PX, col:col + WINDOW_PX]
row, col

In [ ]:
losses_window = {loss: skeleton_recall_loss(p[window], tube[window]) for loss, p in p_track.items()}
losses_image = {loss: skeleton_recall_loss(p, tube) for loss, p in p_track.items()}
losses_window, losses_image

In [ ]:
def densest_zoom(skeleton: np.ndarray) -> np.ndarray:
    """Slice of the `ZOOM_PX` square with the most skeleton pixels inside `skeleton`, on a `ZOOM_PX // 2` grid."""
    h, w = skeleton.shape
    step = ZOOM_PX // 2
    corners = [(r, c) for r in range(0, h - ZOOM_PX + 1, step) for c in range(0, w - ZOOM_PX + 1, step)]
    r, c = max(corners, key=lambda rc: skeleton[rc[0]:rc[0] + ZOOM_PX, rc[1]:rc[1] + ZOOM_PX].sum())
    return np.s_[r:r + ZOOM_PX, c:c + ZOOM_PX]

In [ ]:
zoom = densest_zoom(tube[window])
zoom

# Plots

In [ ]:
MASK_COLOUR = np.array([1.0, 0.85, 0.0])
TUBE_COLOUR = np.array([0.9, 0.1, 0.1])
CENTRELINE_COLOUR = np.array([1.0, 1.0, 1.0])
MASK_GREY = 0.55
COVERAGE_CMAP = "RdYlGn"

In [ ]:
def overlay(rgb: np.ndarray, mask: np.ndarray, colour: np.ndarray, alpha: float) -> np.ndarray:
    out = rgb.astype(float) / 255
    out[mask] = (1 - alpha) * out[mask] + alpha * colour
    return out

In [ ]:
def skeleton_image(track_mask: np.ndarray, tube_mask: np.ndarray, line: np.ndarray) -> np.ndarray:
    """Grey track mask G, red tube S, white one-pixel centreline on black."""
    out = np.zeros(track_mask.shape + (3,))
    out[track_mask] = MASK_GREY
    out[tube_mask] = TUBE_COLOUR
    out[line] = CENTRELINE_COLOUR
    return out

In [ ]:
def coverage_image(p: np.ndarray, tube_mask: np.ndarray) -> np.ndarray:
    """Grey p_track everywhere; on the tube S, p coloured red (missed) to green (covered)."""
    out = np.repeat(p[..., None] * 0.5, 3, axis=2)
    out[tube_mask] = plt.get_cmap(COVERAGE_CMAP)(p[tube_mask])[:, :3]
    return out

In [ ]:
def save_panel(img: np.ndarray, name: str, title: str | None = None) -> None:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(img, interpolation="nearest")
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=12)
    fig.savefig(OUT_DIR / f"{name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.02)
    plt.show()

In [ ]:
rgb = image.rgb[window]
panels = {
    "1-rgb": (rgb / 255, "Супутниковий знімок"),
    "2-track-mask": (overlay(rgb, track[window], MASK_COLOUR, MASK_ALPHA), "Маска дороги G"),
    "3-skeleton": (skeleton_image(track[window], tube[window], centreline[window]), "skel(G) і трубка S"),
    "4-skeleton-zoom": (skeleton_image(track[window][zoom], tube[window][zoom], centreline[window][zoom]),
                        f"Збільшено ({ZOOM_PX}×{ZOOM_PX} px)"),
}
panels.update({f"5-coverage-{loss}": (coverage_image(p_track[loss][window], tube[window]),
                                      f"{loss}: " + "$\\mathcal{L}_{SR}$" + f" = {losses_window[loss]:.2f}")
               for loss in LOSSES})

In [ ]:
for name, (img, title) in panels.items():
    save_panel(img, name, title)

In [ ]:
def save_strip(names: list[str], file_name: str) -> None:
    fig, axes = plt.subplots(1, len(names), figsize=(3.2 * len(names), 3.6))
    for ax, name in zip(axes, names):
        img, title = panels[name]
        ax.imshow(img, interpolation="nearest")
        ax.set_title(title, fontsize=11)
        ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"{file_name}.png", dpi=DPI, bbox_inches="tight", pad_inches=0.05)
    plt.show()

In [ ]:
save_strip(["1-rgb", "2-track-mask", "3-skeleton", "4-skeleton-zoom"], "strip-construction")

In [ ]:
save_strip(["1-rgb", "3-skeleton"] + [f"5-coverage-{loss}" for loss in LOSSES], "strip-coverage")

In [ ]:
def save_colourbar() -> None:
    fig, ax = plt.subplots(figsize=(4, 0.4))
    fig.colorbar(plt.cm.ScalarMappable(cmap=COVERAGE_CMAP), cax=ax, orientation="horizontal",
                 label="$p_{track}$ на скелеті")
    fig.savefig(OUT_DIR / "colourbar.png", dpi=DPI, bbox_inches="tight")
    plt.show()

In [ ]:
save_colourbar()